
# Projeto EBAC + Semantix
# Análise, Segmentação e Previsão de Vendas em um E-commerce Brasileiro

**Objetivo:** identificar padrões de compra, segmentar clientes e prever o valor das vendas utilizando técnicas de Estatística e Machine Learning.


In [ ]:

!pip -q install xgboost plotly seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
from xgboost import XGBRegressor

plt.rcParams['figure.figsize'] = (10,5)


## 1. Carregamento dos Dados

In [ ]:

customers = pd.read_csv('data/olist_customers_dataset.csv')
orders = pd.read_csv('data/olist_orders_dataset.csv')
items = pd.read_csv('data/olist_order_items_dataset.csv')
payments = pd.read_csv('data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('data/olist_order_reviews_dataset.csv')
products = pd.read_csv('data/olist_products_dataset.csv')

for df,name in [(customers,'customers'),(orders,'orders'),(items,'items'),(payments,'payments')]:
    print(name, df.shape)


## 2. Preparação da Base

In [ ]:

base = orders.merge(payments,on='order_id',how='left')
base = base.merge(customers,on='customer_id',how='left')

base['order_purchase_timestamp'] = pd.to_datetime(base['order_purchase_timestamp'])
base['mes'] = base['order_purchase_timestamp'].dt.to_period('M')

base.head()


## 3. Estatística Descritiva

In [ ]:

base['payment_value'].describe()


## 4. Evolução das Vendas

In [ ]:

vendas_mes = base.groupby('mes')['payment_value'].sum()

vendas_mes.plot(kind='line')
plt.title('Receita por Mês')
plt.show()


## 5. Distribuição do Valor das Compras

In [ ]:

sns.histplot(base['payment_value'], bins=50)
plt.show()


## 6. Inferência Estatística

In [ ]:

media = base['payment_value'].mean()

intervalo = stats.t.interval(
    confidence=0.95,
    df=len(base)-1,
    loc=media,
    scale=stats.sem(base['payment_value'])
)

print('Média:', media)
print('IC 95%:', intervalo)


## 7. Segmentação de Clientes

In [ ]:

rfm = base.groupby('customer_unique_id').agg({
    'payment_value':['sum','mean','count']
})

rfm.columns=['valor_total','ticket_medio','qtd_pedidos']
rfm = rfm.reset_index()

scaler = StandardScaler()
X = scaler.fit_transform(rfm[['valor_total','ticket_medio','qtd_pedidos']])


### Método do Cotovelo

In [ ]:

inercia=[]

for k in range(1,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inercia.append(km.inertia_)

plt.plot(range(1,11), inercia)
plt.xlabel('Clusters')
plt.ylabel('Inércia')
plt.show()


### PCA + KMeans

In [ ]:

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters)
plt.title('Clusters de Clientes')
plt.show()

rfm['cluster'] = clusters
rfm.head()


## 8. Machine Learning

In [ ]:

X = rfm[['ticket_medio','qtd_pedidos','cluster']]
y = rfm['valor_total']

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

modelos = {
    'Linear': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
}

resultado=[]

for nome,modelo in modelos.items():
    modelo.fit(X_train,y_train)
    pred = modelo.predict(X_test)

    resultado.append([
        nome,
        mean_absolute_error(y_test,pred),
        np.sqrt(mean_squared_error(y_test,pred)),
        r2_score(y_test,pred)
    ])

resultado = pd.DataFrame(
    resultado,
    columns=['Modelo','MAE','RMSE','R2']
)

resultado.sort_values('R2',ascending=False)


## 9. Cross Validation

In [ ]:

modelo = RandomForestRegressor(random_state=42)

scores = cross_val_score(
    modelo,
    X,
    y,
    cv=5,
    scoring='r2'
)

print(scores)
print('R2 Médio:', scores.mean())



# Conclusão

- Foram identificados padrões de comportamento dos clientes.
- A segmentação permitiu separar grupos de consumidores.
- Os modelos preditivos mostraram potencial para previsão de valor de compra.
- Técnicas de Ciência de Dados podem apoiar decisões estratégicas em e-commerce.
